# Multi-Agent（多智能体系统）
单个 Agent 受到上下文窗口、角色设定和能力边界的限制，难以独立应对极为复杂的工程级任务。通过构建多角色协作的 Multi-Agent 架构，我们可以将复杂的系统拆解为专业化的角色（如需求分析师、架构师、程序员、测试员），实现“分工合作、相互质检、迭代优化”。

## Multi-Agent 架构模式与 LangGraph 多节点通信
1. 理解 Multi-Agent 核心架构模式：深入掌握 网络型（Network）、主管-员工型（Supervisor） 与 层次化/阶梯型（Hierarchical） 的协作模式特点 。
2. 掌握 LangGraph 中的多 Agent 节点通信：学会如何在 State 中共享全局上下文，并通过节点跳转实现 Agent 之间的消息传递与协作
3. 工程实战：构建“研究员 + 撰稿人”双 Agent 协作流水线。


#### Multi-Agent 的三大主流协作架构
1. 主管/路由模式（Supervisor Pattern） ：
    * 原理：设立一个中央主管（Supervisor） Agent。它不直接执行具体业务，而是根据任务进展，动态决定将下一个子任务分配给哪个 Worker Agent 执行（例如：路由给 `Researcher` 或 `Writer`），并在所有 Worker 完成后统一汇总输出 。
    * 适用场景：任务流程明确、各角色职责独立，需要中央控场的场景 。

2. 对等网络模式（Network / P2P Pattern） ：
    * 原理：Agent 之间没有绝对的主次之分，节点根据各自输出的指令直接决定调用下一个节点（如 Agent A 执行完后直接手交任务给 Agent B） 。
    * 适用场景：平级讨论、相互质检与辩论模式（如 Code Review 流程） 。

3. 层次化/树状结构模式（Hierarchical Pattern）
    * 原理：Supervisor 下面还可以嵌套 Sub-Supervisor，形成层层递归的树状团队结构 。
    * 适用场景：极其复杂的全自动软件工程团队（如 MetaGPT/AutoGen 模式） 。

#### 构建“主管-研究员-撰稿员”协作系统
下面我们基于 LangGraph 实现一个标准的 Supervisor Multi-Agent 系统。
* Supervisor 节点：负责任务调度，决定调用 `Researcher` 或 `Writer`，或结束流程（`FINISH`）。
* Researcher 节点：负责收集与梳理资料。
* Writer 节点：负责将研究成果整理为结构化的高质报告。

In [ ]:
from typing import Annotated, Literal, TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from pydantic import BaseModel, Field

# ================= 1. 定义全局共享状态 =================
class MultiAgentState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    next_node: str  # 记录 Supervisor 决定的下一个执行节点

# ================= 2. 定义 Supervisor 路由结构 =================
# 定义 Supervisor 输出的结构化 Pydantic 模型
class Router(BaseModel):
    """选择下一个应该工作的 Agent 节点，或者完成任务。"""
    next_action: Literal["Researcher", "Writer", "FINISH"] = Field(
        description="选择 'Researcher' 搜集资料，'Writer' 撰写报告，若任务已完成则选择 'FINISH'"
    )

llm = ChatOpenAI(model="gpt-4o", temperature=0)

# 使用 structured_output 让 Supervisor 产生精准的路由指令
supervisor_llm = llm.with_structured_output(Router)

def supervisor_node(state: MultiAgentState):
    """中央主管节点：判断当前任务进度，决定下一个干活的角色"""
    system_prompt = (
        "你是一个项目主管，负责管理团队协作。"
        "团队成员包括：\n"
        "1. Researcher: 负责收集和分析主题相关的信息与资料。\n"
        "2. Writer: 负责基于 Researcher 提供的资料编写逻辑严密的最终报告。\n"
        "请根据当前的对话历史，指定下一个干活的角色。如果 Writer 已经完成了报告，请选择 FINISH。"
    )

    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = supervisor_llm.invoke(messages)

    # 将决策结果存入 state 中的 next_node 字段
    return {"next_node": response.next_action}

# ================= 3. 定义 Worker 节点 =================
def researcher_node(state: MultiAgentState):
    """研究员节点：搜集资料"""
    system_prompt = "你是专业的研究员。请针对用户提出的主题，列出 3 个核心要点和深度背景信息。"
    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = llm.invoke(messages)

    # 标注消息来自于 Researcher
    return {
        "messages": [HumanMessage(content=response.content, name="Researcher")]
    }

def writer_node(state: MultiAgentState):
    """撰稿员节点：整理撰写报告"""
    system_prompt = "你是资深撰稿员。请根据团队中 Researcher 提供的研究要点，将其整理成一篇结构清晰的 Markdown 格式报告。"
    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = llm.invoke(messages)

    # 标注消息来自于 Writer
    return {
        "messages": [HumanMessage(content=response.content, name="Writer")]
    }

# ================= 4. 构建 Supervisor StateGraph 图 =================
workflow = StateGraph(MultiAgentState)

# 添加节点
workflow.add_node("Supervisor", supervisor_node)
workflow.add_node("Researcher", researcher_node)
workflow.add_node("Writer", writer_node)

# 设置起始节点为 Supervisor
workflow.add_edge(START, "Supervisor")

# 添加 Supervisor 节点的条件路由（条件边）
workflow.add_conditional_edges(
    "Supervisor",
    lambda state: state["next_node"], # 从 state["next_node"] 读取下一个目标
    {
        "Researcher": "Researcher",
        "Writer": "Writer",
        "FINISH": END
    }
)

# 每个 Worker 执行完任务后，都将结果吐回给 Supervisor 重新审查调度
workflow.add_edge("Researcher", "Supervisor")
workflow.add_edge("Writer", "Supervisor")

app = workflow.compile()

# ================= 5. 执行多智能体协作流程 =================
initial_state = {
    "messages": [HumanMessage(content="请帮我撰写一份关于‘大模型 Agent 在医疗领域应用’的简报。")]
}

print("🚀 启动 Multi-Agent 团队协作系统...\n")

for event in app.stream(initial_state):
    for node_name, output in event.items():
        print(f"------------ 节点 [{node_name}] 输出 ------------")
        if "next_node" in output:
            print(f"🎯 Supervisor 指派下一个节点: {output['next_node']}")
        if "messages" in output:
            latest_msg = output["messages"][-1]
            print(f"💬 [{latest_msg.name or 'User'}]:\n{latest_msg.content[:150]}...\n")

1. 路由决策解耦：通过 `with_structured_output(Router)` 约束 LLM 强制返回固定的枚举值 (`Researcher` / `Writer` / `FINISH`)，避免大模型输出自由文本导致分支判断失败 。
2. 全局状态与 Context 共享：在 LangGraph 中，所有 Worker Agent 都向公共的 `messages` 列表中追加消息（使用 `add_messages` 规约）。因此，`Writer` 节点天生能够看到 `Researcher` 在上一轮产出的所有研究成果 。
3. `name` 属性的作用：在构造返回消息时设置 `name="Researcher"` / `name="Writer"`，这能让大模型在后续轮次中清晰地识别哪句话是哪个专业角色说出来的，大幅降低角色混淆 。
4. 死循环防御：如果 `Supervisor` 陷入死循环，不断在 `Researcher` 和 `Writer` 之间来回指派，你应该如何在 State 中增加一个 `recursion_count` 或最大步数限制来防护？
5. 人工干预扩展：结合`Human-in-the-Loop` 机制，尝试在 `Supervisor` 做出 `FINISH` 决策时插入一个 `interrupt_before` 中断，由人类审查员决定是否“批准结案”还是“打回重写”。

## Peer-to-Peer 去中心化协作网络
Supervisor（主管-员工）是最经典且控场能力强的 Multi-Agent 架构。然而在实际开发中，如果所有任务都需要过一遍中央主管，系统往往会产生额外的 Token 开销和响应延迟；且在涉及多角色质检、辩论与交接棒的场景下，集中控场反而会限制智能体的发挥。

去中心化网状协作模式让 Agent 之间像接力赛一样“手递手传球”，通过直接对话与交叉质检（Cross-Review）完成复杂的迭代任务。
1. 掌握 Peer-to-Peer 协作机制：理解无中央主管（Supervisor-less）拓扑结构下，Agent 如何自主判断“下一步手递手交给谁”。
2. 掌握交叉质检与自我纠错（Cross-Review Loop）：构建由“生成者（Generator）”与“审查者（Reviewer）”组成的死循环纠错双星系统。
3. 基于 LangGraph 手写一个“代码编写员 + 代码审查员”协同纠错系统。

#### Peer-to-Peer 模式与 Supervisor 模式对比
| 维度     | Supervisor 模式 (中央主管)               | Peer-to-Peer 模式 (去中心化网状)           |
|--------|------------------------------------|------------------------------------|
| 控制中心   | 存在统一的中央决策节点（Router）。               | 无中央节点，由 Agent 自身决定接收人。             |
| 数据与流转  | Worker 必须将结果汇报给 Supervisor，再由主管分发。 | Agent A 完成后，在输出消息中直接指定下一站为 Agent B |
| 典型适用场景 | 角色多、路径不确定、需要全局控场的分流任务。             | 角色间关系紧密、依赖极强的链式质检、辩论与循环迭代。         |
| 通讯效率   | 路径跳转较多，可能会多消耗一轮调度 Token。           | 跳转直截了当，通信链路更紧凑直接。                  |

**P2P 路由的核心魔法（Hand-off Mechanism）：**
在 LangGraph 中实现 P2P 传球，主要依赖各个 Agent 节点在返回更新时，附带一个角色指令/交接信号。通常可以使用 `Command(goto="Coder")` 或自定义在 `State` 中的 `active_agent` 字段，实现直接无缝跳转。

#### 构建“代码编写员 + 严苛审查员”自纠错闭环
下面我们基于 LangGraph 实现一个标准的 P2P 协作网络。
* Coder 节点：负责根据需求编写 Python 代码；如果收到 Reviewer 的修改意见，则根据意见重构代码。
* Reviewer 节点：负责对 Coder 的代码进行静态审查与逻辑推演。如果发现 Bug 或优化点，直接打回给 Coder 并附带意见；只有当代码完全无误时，才输出 `PASS` 并结束流程。

In [ ]:
from typing import Annotated, Literal, TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# ================= 1. 定义全局状态 =================
class P2PState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    review_status: str  # 记录审查状态: "APPROVED" 或 "NEEDS_REVISION"

llm = ChatOpenAI(model="gpt-4o", temperature=0)

# ================= 2. 定义 Coder 节点 (程序员) =================
def coder_node(state: P2PState):
    """程序员 Agent：编写代码或根据审查意见进行修正"""
    system_prompt = (
        "你是一名资深的 Python 开发工程师。"
        "如果这是初始需求，请编写符合要求的规范 Python 代码。\n"
        "如果对话历史中包含 Reviewer 的修改意见，请仔细分析意见并对其提出的 Bug 进行重构修复。"
        "请务必把编写好的代码放入 ```python Markdown 代码块中。"
    )

    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = llm.invoke(messages)

    # 手递手将带有名字的消息放入 State，并重置审阅状态
    return {
        "messages": [HumanMessage(content=response.content, name="Coder")],
        "review_status": "NEEDS_REVISION"
    }

# ================= 3. 定义 Reviewer 节点 (代码审查员) =================
def reviewer_node(state: P2PState):
    """审查员 Agent：严格评审 Coder 的代码"""
    system_prompt = (
        "你是一名极其严苛的代码评审专家 (Code Reviewer)。\n"
        "你的职责是检查 Coder 最新提交的代码是否存在以下问题：\n"
        "1. 边界条件缺陷 (如除以零、空输入处理)\n"
        "2. 算法时间/空间复杂度不优\n"
        "3. 代码风格与可读性问题\n\n"
        "【输出规则】:\n"
        "- 如果发现任何缺陷或改进空间，请详细指出问题位置并提供修改建议，且**不要**包含 'APPROVED' 关键字。\n"
        "- 如果代码完美无瑕、可直接上线，请在你的回复中包含大写的单词 'APPROVED'！"
    )

    messages = [SystemMessage(content=system_prompt)] + state["messages"]
    response = llm.invoke(messages)

    # 检查 Reviewer 的回复中是否包含批准标记
    if "APPROVED" in response.content.upper():
        status = "APPROVED"
    else:
        status = "NEEDS_REVISION"

    return {
        "messages": [HumanMessage(content=response.content, name="Reviewer")],
        "review_status": status
    }

# ================= 4. 定义 P2P 路由逻辑 (核心!) =================
def route_review_result(state: P2PState) -> Literal["Coder", "__end__"]:
    """
    路由决策：直接根据审查结果决定是“打回给 Coder 重写”还是“直接结案”
    这就是去中心化的网状传球（无需 Supervisor 参与）
    """
    if state["review_status"] == "APPROVED":
        return END
    return "Coder"

# ================= 5. 构建 P2P 状态图 =================
workflow = StateGraph(P2PState)

# 添加对等节点
workflow.add_node("Coder", coder_node)
workflow.add_node("Reviewer", reviewer_node)

# 起始直接走向 Coder
workflow.add_edge(START, "Coder")

# 【P2P 手递手传球 1】Coder 写完代码后，直接无条件传球给 Reviewer 审查
workflow.add_edge("Coder", "Reviewer")

# 【P2P 手递手传球 2】Reviewer 审查后，通过条件边决定是打回给 Coder 还是结束
workflow.add_conditional_edges(
    "Reviewer",
    route_review_result,
    {
        "Coder": "Coder",  # 没通过审查：直接传回 Coder 打回重写（形成闭环）
        END: END           # 审查通过：结案
    }
)

app = workflow.compile()

# ================= 6. 运行 P2P 质检系统 =================
initial_state = {
    "messages": [HumanMessage(content="请帮我写一个 Python 函数：实现两个大整数相除的字符串格式计算，要考虑各种边界极端情况。")]
}

print("🚀 启动 Peer-to-Peer 程序员与审查员双星纠错网络...\n")

for event in app.stream(initial_state, config={"recursion_limit": 10}):
    for node_name, output in event.items():
        print(f"================ 节点 [{node_name}] 输出 ================")
        latest_msg = output["messages"][-1]
        print(f"💬 [{latest_msg.name}]:\n{latest_msg.content[:200]}...\n")
        if "review_status" in output:
            print(f"📊 当前审查判定: {output['review_status']}\n")

1. 去中心化的闭环链路：`Coder -> Reviewer -> (若未通过) -> Coder` 构成了一个自我纠错与迭代的环路（Loop）。两者直接以对话历史作为上下文相互质检，脱离了 Supervisor 的中转。
2. 状态驱动的动态分支：我们将 `review_status` 作为显示的状态标记注入 `P2PState`，并通过简短的条件判断函数 `route_review_result` 直接控制流向，不仅降低了逻辑复杂度，也保证了跳转的确定性。
3. 递归深度防御 (`recursion_limit`)：在 P2P 的死循环网络中，如果 Reviewer 要求过于苛刻，容易导致死循环。因此在 `app.stream()` 时传入 `config={"recursion_limit": 10}` 是极佳的生产工程防线，避免 Token 无上限地消耗。
4. 思考：多节点 P2P 扩展：如果在 `Coder` 与 `Reviewer` 之间，再加入一个 `Tester`（测试员 Agent，专门生成 Unittest 并模拟运行代码），你将如何设计三者之间“手递手”的传递路径（例如 `Coder -> Tester -> Reviewer`）？
5. 角色冲撞问题：在极长的 P2P 对话中，`messages` 会不断增长，`Coder` 和 `Reviewer` 可能会因为历史信息过多而逐渐搞混自己的身份（即产生“角色漂移”）。除了在节点中加入 `SystemMessage` 之外，有什么机制能在传球前对历史消息进行针对性的上下文裁切？